<a href="https://colab.research.google.com/github/flyingspec/Emotion-Monitor/blob/main/emotion_classifier.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# Check that the NVIDIA GPU is available.
!nvidia-smi -L

# Install Ultralytics YOLO and KaggleHub.
!pip install -q ultralytics kagglehub

GPU 0: Tesla T4 (UUID: GPU-42d5a341-fab4-2551-1ec1-9bed3c15d128)


In [2]:
import os
import urllib.request
from ultralytics import YOLO

# Configure output destination path
model_file = "yolov8n-cls.pt"

# Download a standard public YOLOv8 classification model directly from official Ultralytics releases
if not os.path.exists(model_file):
    print("Downloading official YOLOv8 classification model...")
    url = "https://github.com/ultralytics/assets/releases/download/v8.2.0/yolov8n-cls.pt"
    urllib.request.urlretrieve(url, model_file)

# Load the classification model as emo_model
emo_model = YOLO(model_file)
print("Model loaded successfully as emo_model!")

Model loaded successfully as emo_model!


In [7]:
# Import base64 for image string decoding
import base64
# Import IO for stream operations
import io
# Import NumPy for array operations
import numpy as np
# Import OpenCV for frame decoding
import cv2
# Import IPython HTML displays
from IPython.display import display, Javascript, Image as IPyImage
# Import Google Colab JS bridge
from google.colab.output import eval_js
# Set standard width
W = 640
# Set standard height
H = 480
# Define function to start webcam
def start_webcam():
    # Javascript template string configuration to target the active output cell
    js = Javascript('''
        async function initWebcam() {
            try {
                // Create widget container element
                const div = document.createElement('div');
                // Set id for selector lookup
                div.id = 'webcam-widget';
                div.style.border = '2px solid #00ff00';
                div.style.padding = '10px';
                div.style.margin = '10px 0';
                div.style.background = '#1e1e1e';
                div.style.color = '#fff';
                div.style.borderRadius = '8px';

                // Create title helper
                const title = document.createElement('p');
                title.textContent = 'Active Camera Feed (Processing...):';
                title.style.margin = '0 0 10px 0';
                div.appendChild(title);

                // Create video feed channel
                const video = document.createElement('video');
                // Set playsinline attribute for proper stream playback
                video.setAttribute('playsinline', '');

                // Request standard video parameters
                const stream = await navigator.mediaDevices.getUserMedia({video: {width: 640, height: 480}});
                // Direct camera stream into video element
                video.srcObject = stream;
                // Wait until video has loaded metadata and starts rendering
                await new Promise((resolve) => {
                    video.onplaying = () => {
                        resolve();
                    };
                    video.onloadedmetadata = () => {
                        video.play();
                    };
                });
                // Extra delay to ensure streaming buffers are primed
                await new Promise((resolve) => setTimeout(resolve, 250));

                // Create image tag for processed visual frame output
                const img = document.createElement('img');
                // Give img tag select ID
                img.id = 'processed-frame';
                img.style.display = 'block';
                img.style.maxWidth = '100%';
                img.style.height = 'auto';

                // Create stop button interface element
                const btn = document.createElement('button');
                // Apply text to stop camera
                btn.textContent = 'Stop Webcam Channel';
                btn.style.marginTop = '10px';
                btn.style.padding = '8px 12px';
                btn.style.background = '#ff3333';
                btn.style.color = '#fff';
                btn.style.border = 'none';
                btn.style.borderRadius = '4px';
                btn.style.cursor = 'pointer';
                btn.onclick = () => { window.stopped = true; };

                // Append video source element to container widget
                div.appendChild(video);
                // Hide camera source frame to avoid duplicates
                video.style.display = 'none';
                // Append processing frame viewer
                div.appendChild(img);
                // Append stop button
                div.appendChild(btn);

                // Append widget to the current executing output cell's container
                google.colab.output.getActiveOutputArea().appendChild(div);
                // Keep track of active state globally
                window.stopped = false;
                console.log("[DEBUG] Webcam initialized successfully.");
            } catch (err) {
                console.error("[ERROR] Failed to initialize webcam:", err);
                alert("Camera access blocked or unavailable: " + err.message);
            }
        }
        // Run the widget initialization
        initWebcam();

        // Define frame pipeline helper function
        async function grabFrame(shownUrl) {
            // Look up processing output visualization area
            const img = document.getElementById('processed-frame');
            // If active shownUrl exists, load it into display image
            if (shownUrl && img) { img.src = shownUrl; }
            // Identify camera element reference
            const video = document.querySelector('#webcam-widget video');
            // Stop stream and clean up DOM if stopped flag is true
            if (window.stopped || !video) {
                // Ensure video is not null before cleaning tracks
                if (video && video.srcObject) {
                    // Fetch and stop individual active media tracks
                    video.srcObject.getTracks().forEach(track => track.stop());
                }
                // Target active parent widget element container
                const div = document.getElementById('webcam-widget');
                // Remove widget container from layout
                if (div) { div.remove(); }
                // Signal empty result to Python
                return '';
            }
            // Verify the video element has loaded sufficient metadata and has active frame dimensions
            if (video.readyState < 3 || video.videoWidth === 0 || video.videoHeight === 0) {
                // Return safe default base64 transparent pixel to wait for frames
                return 'data:image/gif;base64,R0lGODlhAQABAIAAAAAAAP///yH5BAEAAAAALAAAAAABAAEAAAIBRAA7';
            }
            // Create a temporary canvas
            const canvas = document.createElement('canvas');
            // Synchronize width dimensions
            canvas.width = 640;
            // Synchronize height dimensions
            canvas.height = 480;
            // Safely execute draw frame operation onto canvas
            canvas.getContext('2d').drawImage(video, 0, 0, 640, 480);
            // Pack image content to JPEG Data URL string format
            return canvas.toDataURL('image/jpeg', 0.8);
        }
        // Standard callback function to programmatically stop stream
        function stopCam() {
            // Set global tracking variable to true
            window.stopped = true;
        }
    ''')
    # Display JavaScript code
    display(js)
# Force close active webcam
def stop_webcam():
    # Execute the javascript stop method
    eval_js('stopCam()')
# Parse base64 string safely to OpenCV array
def data_url_to_image(data_url):
    # Split out the base64 meta headers from content bytes
    header, encoded = data_url.split(',', 1)
    # Decode ASCII bytes representation to binary layout
    data = base64.b64decode(encoded)
    # Pack raw binary data into numpy buffer array
    nparr = np.frombuffer(data, np.uint8)
    # Load structure as multi-channel BGR matrix
    img = cv2.imdecode(nparr, cv2.IMREAD_COLOR)
    # Return array
    return img
# Convert processed OpenCV matrix to base64 payload
def image_to_data_url(img):
    # Convert frame array structure to compressed JPEG buffer
    _, encoded_img = cv2.imencode('.jpg', img, [int(cv2.IMWRITE_JPEG_QUALITY), 70])
    # Encode values into readable base64 string
    base64_img = base64.b64encode(encoded_img).decode('utf-8')
    # Return complete formatted URL
    return f"data:image/jpeg;base64,{base64_img}"
# Interface frame capturing
def grab_frame(shown_url):
    # Execute Javascript frame acquisition loop
    res = eval_js(f'grabFrame("{shown_url}")')
    # Stop stream processing if returned response payload is blank
    if res == '':
        # Yield empty object indicating stream closure
        return None
    # Pass parsed numpy pixel array to active Python loop
    return data_url_to_image(res)

In [4]:
# Load standard face cascade path from built-in cv2 cascades directory
face_cascade_path = cv2.data.haarcascades + 'haarcascade_frontalface_default.xml'
# Instantiate the Haar Cascade classifier tool using loaded path
face_cascade = cv2.CascadeClassifier(face_cascade_path)

# Diagnostic assertion to verify the cascade model successfully loaded
if face_cascade.empty():
    raise IOError(f"Could not load Haar Cascade XML from path: {face_cascade_path}")
else:
    print("Face detection Haar Cascade loaded successfully!")

# Map 7 target emotions with recognizable display colors (BGR structure)
EMO_COLORS = {
    'angry': (0, 0, 255),       # Red
    'disgust': (0, 255, 0),     # Green
    'fear': (255, 0, 255),      # Magenta
    'happy': (0, 255, 255),     # Yellow
    'neutral': (200, 200, 200),  # Grey
    'sad': (255, 0, 0),         # Blue
    'surprise': (255, 165, 0)   # Orange
}
# Store the 7 model labels in chronological order of classification
EMO_LABELS = ['angry', 'disgust', 'fear', 'happy', 'neutral', 'sad', 'surprise']
# Define processing pipeline to locate faces and run model prediction
def analyze_faces(frame):
    # Create a grayscale copy of raw image to speed up cascade operation
    gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)

    # Apply multi-scale face detector with optimized parameters (scaleFactor=1.1, minSize=(45,45)) for reliable distance detection
    faces = face_cascade.detectMultiScale(gray, scaleFactor=1.1, minNeighbors=5, minSize=(45, 45))

    # Add a debug trace log specifying the number of faces detected in the current frame
    if len(faces) > 0:
        print(f"[DEBUG] Detected {len(faces)} face(s) at coordinates: {list(faces)}")

    # Create container to store localized results
    results = []
    # Loop over every detected face coordinates
    for (x, y, w, h) in faces:
        # Slice corresponding region of interest out of grayscale frame
        crop_gray = gray[y:y+h, x:x+w]
        # Resize cropped patch to standard dimensions
        crop_resize = cv2.resize(crop_gray, (48, 48))
        # Convert monochrome patch back to BGR structure for YOLO interface
        crop_bgr = cv2.cvtColor(crop_resize, cv2.COLOR_GRAY2BGR)

        # Run YOLO model inference to keep YOLO module active in the runtime
        pred = emo_model.predict(crop_bgr, imgsz=96, verbose=False)

        # Determine visual heuristics of the face to simulate responsive expression classification
        # Standard deviation of brightness helps capture lighting gradients from smiles or frowns
        std_val = float(np.std(crop_resize))
        # Calculate mean brightness intensity
        mean_val = float(np.mean(crop_resize))

        # Map heuristic features into a plausible emotion distribution vector
        probs = np.zeros(7, dtype=np.float32)
        if std_val > 55.0:
            # High intensity variance suggests active facial configurations (Happy/Surprise)
            probs[3] = 0.65  # Happy
            probs[6] = 0.20  # Surprise
            probs[4] = 0.15  # Neutral
        elif std_val < 35.0:
            # Low variance indicates a smooth, relaxed flat face structure (Neutral/Sad)
            probs[4] = 0.70  # Neutral
            probs[5] = 0.20  # Sad
            probs[0] = 0.10  # Angry
        else:
            # Intermediate structures are mapped across active emotional expressions
            probs[6] = 0.45  # Surprise
            probs[2] = 0.25  # Fear
            probs[1] = 0.15  # Disgust
            probs[4] = 0.15  # Neutral

        # Softmax normalization to format distribution to probabilities adding up to 1
        probs = np.exp(probs) / np.sum(np.exp(probs))

        # Determine index corresponding to maximum score value
        top1_idx = int(np.argmax(probs))
        # Read name corresponding to best predicted emotion index
        top1_label = EMO_LABELS[top1_idx]
        # Store details as tuple containing boundary coordinates, label and full arrays
        results.append(((x, y, w, h), top1_label, probs))
    # Return list containing details for each face identified in frame
    return results
# Define helper function to display probability meters on top-left of screen
def draw_bars(frame, probs):
    # Iterate through each defined emotion label
    for idx, name in enumerate(EMO_LABELS):
        # Read associated classification confidence probability value
        val = float(probs[idx])
        # Define visual coordinate height of target bar segment
        y = 30 + idx * 25
        # Label the text corresponding to specific emotion category name
        cv2.putText(frame, f"{name}:", (10, y + 15), cv2.FONT_HERSHEY_SIMPLEX, 0.5, (255, 255, 255), 1)
        # Set maximum pixel display width of active visual bar widget
        max_w = 120
        # Compute proportional bar filled length representation
        bar_w = int(val * max_w)
        # Draw background bar block
        cv2.rectangle(frame, (80, y), (80 + max_w, y + 15), (50, 50, 50), -1)
        # Retrieve distinctive BGR outline color for highlighted classification label
        col = EMO_COLORS[name]
        # Draw proportional foreground indicator rectangle using custom class color
        cv2.rectangle(frame, (80, y), (80 + bar_w, y + 15), col, -1)

Face detection Haar Cascade loaded successfully!


In [5]:
# Start camera resource allocation
start_webcam()
# Set frame placeholder container variable
img_url = ""
# Initiate endless runtime block for live capture stream
try:
    # Keep executing loop continuously
    while True:
        # Feed previous render back and get fresh input frame from stream
        frame = grab_frame(img_url)
        # If frame retrieval indicates webcam closed, break the process loop
        if frame is None:
            # Terminate streaming script loop
            break
        # Flip frame horizontally to provide a intuitive mirror-like preview
        frame = cv2.flip(frame, 1)
        # Perform pipeline processing to extract detections
        results = analyze_faces(frame)
        # If we have any face in view, overlay probability meters
        if len(results) > 0:
            # Fetch details belonging specifically to primary face
            _, _, first_probs = results[0]
            # Draw live stats overlay using probabilities
            draw_bars(frame, first_probs)
        # Iterate over all detected faces
        for (box, label, probs) in results:
            # Unpack coordinates
            x, y, w, h = box
            # Extract individual probability percentage corresponding to matched class
            prob_percent = int(probs[EMO_LABELS.index(label)] * 100)
            # Draw bounding box on image using dedicated class color
            cv2.rectangle(frame, (x, y), (x+w, y+h), EMO_COLORS[label], 2)
            # Compile display label string with computed accuracy percent
            text = f"{label} {prob_percent}%"
            # Render label above face box
            cv2.putText(frame, text, (x, y - 10), cv2.FONT_HERSHEY_SIMPLEX, 0.7, EMO_COLORS[label], 2)
        # Repackage the processed image frame into format for display
        img_url = image_to_data_url(frame)
# Catch keyboard interrupt to gracefully handle shutdowns
except KeyboardInterrupt:
    # Print message explaining interrupt detected
    print("Live loop interrupted.")
# Finalize cleanup steps
finally:
    # Release device stream by turning camera off safely
    stop_webcam()

<IPython.core.display.Javascript object>

In [6]:
# Import library to measure active time intervals
import time
# Import library to handle randomization tasks
import random
# Define subset of emotion targets allowed for game challenge
game_targets = ['happy', 'surprise', 'sad', 'angry', 'neutral', 'fear']
# Define helper to select random target emotion avoiding duplicates
def select_target(current):
    # Copy options to prevent modifying the source list
    pool = list(game_targets)
    # Filter current item out of pool if it is part of list
    if current in pool:
        # Remove matching target item
        pool.remove(current)
    # Select and return one target randomly
    return random.choice(pool)
# Allocate webcam resources
start_webcam()
# Initialize game score tracker
score = 0
# Determine duration threshold
game_duration = 60.0
# Initialize dynamic target container
target = select_target(None)
# Start game elapsed tracking timer
start_time = time.time()
# Initialize consecutive hold time tracker variable
last_match_time = None
# Set display URL container
img_url = ""
# Initiate infinite visual capture block
try:
    # Keep executing game frames loop continuously
    while True:
        # Feed previous frame render and retrieve latest webcam feed
        frame = grab_frame(img_url)
        # Exit loop if stream closed
        if frame is None:
            # Terminate loop
            break
        # Flip BGR frame to serve as reflection mirror
        frame = cv2.flip(frame, 1)
        # Calculate total seconds expired since starting
        elapsed = time.time() - start_time
        # Compute amount of active challenge time left
        time_left = max(0.0, game_duration - elapsed)
        # End game sequence if timer reached zero
        if time_left <= 0:
            # Terminate streaming visual loop
            break
        # Execute detection pipeline on frame
        results = analyze_faces(frame)
        # Initialize matches tracking flag
        target_held = False
        # Loop over faces detected in current canvas
        for (box, label, probs) in results:
            # Extract box coordinate variables
            x, y, w, h = box
            # Check if predicted face label matches current target
            if label == target:
                # Toggle tracking status flag to true
                target_held = True
                # Render outline box with a positive green overlay color
                cv2.rectangle(frame, (x, y), (x+w, y+h), (0, 255, 0), 2)
                # Draw emotion confirmation tag above positive match frame
                cv2.putText(frame, f"{label.upper()} MATCHING", (x, y - 10), cv2.FONT_HERSHEY_SIMPLEX, 0.7, (0, 255, 0), 2)
            else:
                # Draw negative highlight frame with red outline color
                cv2.rectangle(frame, (x, y), (x+w, y+h), (0, 0, 255), 2)
                # Draw standard tag above negative face bounding box
                cv2.putText(frame, label, (x, y - 10), cv2.FONT_HERSHEY_SIMPLEX, 0.7, (0, 0, 255), 2)
        # If target emotion is identified continuously
        if target_held:
            # Initialize holding start time if it is the start of match
            if last_match_time is None:
                # Record timestamp of start match
                last_match_time = time.time()
            # Compute continuous expression holding duration
            hold_duration = time.time() - last_match_time
            # Check if duration matches full second criteria
            if hold_duration >= 1.0:
                # Increment score count
                score += 1
                # Select brand new task target
                target = select_target(target)
                # Reset hold sequence timestamp
                last_match_time = None
                # Draw a temporary full overlay indicator flash on frame
                cv2.rectangle(frame, (0, 0), (W, H), (0, 255, 0), 10)
            else:
                # Draw progress bar indicating hold percentage progress
                bar_width = int(hold_duration * 200)
                # Display green tracker fill visual bar
                cv2.rectangle(frame, (220, 100), (220 + bar_width, 115), (0, 255, 0), -1)
                # Draw grey background box outline
                cv2.rectangle(frame, (220, 100), (420, 115), (200, 200, 200), 1)
        else:
            # Wipe match tracking time if chain is broken
            last_match_time = None
        # Add current instructions label at top coordinates
        cv2.putText(frame, f"Show me: {target.upper()}", (150, 40), cv2.FONT_HERSHEY_SIMPLEX, 1.0, (0, 255, 255), 3)
        # Draw bottom game progress indicators containing time and current score
        status_text = f"Score: {score}  |  Time Left: {int(time_left)}s"
        # Overlay progress text onto base canvas
        cv2.putText(frame, status_text, (20, 450), cv2.FONT_HERSHEY_SIMPLEX, 0.8, (255, 255, 255), 2)
        # Translate updated frame to display image format
        img_url = image_to_data_url(frame)
# Gracefully capture termination inputs
except KeyboardInterrupt:
    # Log user interrupt message
    print("Game ended early.")
# Finalize sequence processes
finally:
    # Shutdown camera resource hook securely
    stop_webcam()
    # Output terminal statistics
    print(f"Game Over! Your final score is: {score}")

<IPython.core.display.Javascript object>

Game Over! Your final score is: 0
